In [1]:
# reprendre nettoyage

import sys
sys.path.append('..')

from utils.fonctions import load_parquet_data, normalize_colnames_list
from src.artifacts.data_processing import Nettoyage

TARGET_COL = 'consommation_annuelle_moyenne_par_logement_de_l_adresse_kwh_enedis_with_ban'

In [13]:
# custom functions
get_conso_colnames = lambda data: list(filter(lambda c: 'conso' in c, data.columns))
get_deperdition_colnames = lambda data: list(filter(lambda c: 'deperdition' in c, data.columns))
get_dpe_colnames = lambda data: list(filter(lambda c: 'dpe' in c, data.columns))



def numerize(data, encoding_pipeline, downcast_num_cols=True):

    def downcast_numeric_cols():
        # int32 range = 2**32/2 à gauche et à droite de 0
        for c in data.columns:
            if data[c].dtype == 'int64':
                data[c] = data[c].astype('int32')
            elif data[c].dtype == 'float64':
                data[c] = data[c].astype('float32')
        return data
    
    encoding_pipeline.set_output(transform='pandas')
    data = encoding_pipeline.fit_transform(data)

    if downcast_num_cols:
        return downcast_numeric_cols()
    return data

In [3]:
df = load_parquet_data('../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet')

pipeline = Nettoyage(df)

pipeline.run(use_entropy_selection = True)
df_clean_v1 = pipeline.df

del df

Loading parquet data from : ../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet..
-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 250 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 55 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 146 colonnes et 375231 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['emission_ges_chauffage_energie_ndeg1_ademe', 'conso_chauffage_installation_chauffage_ndeg1_ademe', 'emission_ges_5_usages_energie_ndeg1_ademe', 'conso_ecs_e_primaire_ademe', 'emission_ges_eclairage_ademe', 'conso_e_finale_depensier_generateur_ecs_ndeg1_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'conso_chauffage_generateur_ndeg1_installation_ndeg1_ademe', 'emission_ges_ecs_ademe', 'e

In [6]:
df_clean_v1.dtypes.value_counts()

float32           50
string[python]    39
Name: count, dtype: int64

In [17]:
df = load_parquet_data('../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet')

pipeline = Nettoyage(df)

pipeline.run(use_target_correlation_selection=True)
df_clean_v2 = pipeline.df

del df

Loading parquet data from : ../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet..


-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 250 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 55 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 146 colonnes et 375231 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['emission_ges_chauffage_energie_ndeg1_ademe', 'conso_chauffage_depensier_installation_chauffage_ndeg1_ademe', 'conso_chauffage_installation_chauffage_ndeg1_ademe', 'emission_ges_5_usages_energie_ndeg1_ademe', 'cout_refroidissement_depensier_ademe', 'emission_ges_eclairage_ademe', 'conso_e_finale_depensier_generateur_ecs_ndeg1_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'conso_chauffage_generateur_ndeg1_installation_ndeg1_ademe', 'emission_ges_ecs_ademe', 'emission_ges_ecs_depensier_ademe', 

In [18]:
df_clean_v2.dtypes.value_counts()

float32           50
string[python]    39
Name: count, dtype: int64

In [8]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer

### Encodage dataframe v1 (use entropy selection)

*A ce niveau la partie nettoyage de data est finie (version mvp) est ok. On va faire un pieplien d'encoding ici.*

In [5]:
df_clean_v1[df_clean_v1.isna().sum().loc[df_clean_v1.isna().sum().values > 0].index].dtypes

# les na dans les col de type non numeric (exple string)
# et garder les dates ?

configuration_installation_chauffage_ndeg1_ademe          string[python]
configuration_installation_ecs_ademe                      string[python]
type_installation_chauffage_ndeg1_ademe                   string[python]
type_installation_ecs_general_ademe                       string[python]
usage_generateur_ecs_ndeg1_ademe                          string[python]
type_energie_generateur_ecs_ndeg1_ademe                   string[python]
type_generateur_ecs_ndeg1_ademe                           string[python]
logement_traversant_0_1_ademe                             string[python]
qualite_isolation_murs_ademe                              string[python]
type_emetteur_installation_chauffage_ndeg1_ademe          string[python]
classe_inertie_batiment_ademe                             string[python]
indicateur_confort_ete_ademe                              string[python]
type_energie_ndeg2_ademe                                  string[python]
type_installation_ecs_ademe                        

In [6]:
# type string
df_clean_v1.select_dtypes(include='string').head() 

,configuration_installation_chauffage_ndeg1_ademe,configuration_installation_ecs_ademe,type_installation_chauffage_ndeg1_ademe,type_installation_ecs_general_ademe,usage_generateur_ecs_ndeg1_ademe,type_energie_generateur_ecs_ndeg1_ademe,type_generateur_ecs_ndeg1_ademe,logement_traversant_0_1_ademe,qualite_isolation_menuiseries_ademe,qualite_isolation_murs_ademe,...,etiquette_ges_ademe,etiquette_dpe_ademe,type_generateur_ndeg1_installation_ndeg1_ademe,qualite_isolation_plancher_bas_ademe,type_energie_principale_ecs_ademe,qualite_isolation_plancher_haut_toit_terrase_ademe,categorie_enr_ademe,appartement_non_visite_0_1_ademe,type_enedis_with_ban,arrondissement
0,Installation de chauffage simple,Un seul système d'ECS sans solaire,installation individuelle,individuel,chauffage + ecs,Gaz naturel,Chaudière gaz à condensation après 2015,True,très bonne,insuffisante,...,C,C,Chaudière gaz à condensation après 2015,très bonne,Gaz naturel,<NA>,<NA>,<NA>,housenumber,6
1,Installation de chauffage simple,Un seul système d'ECS sans solaire,installation individuelle,individuel,ecs,Électricité,Ballon électrique à accumulation vertical Caté...,False,insuffisante,insuffisante,...,B,D,Radiateur électrique à accumulation,très bonne,Électricité,<NA>,<NA>,<NA>,housenumber,6
2,Installation de chauffage simple,Un seul système d'ECS sans solaire,installation individuelle,individuel,ecs,Électricité,Ballon électrique à accumulation horizontal,True,insuffisante,insuffisante,...,D,E,Chaudière gaz basse température 2001-2015,très bonne,Électricité,<NA>,<NA>,<NA>,housenumber,6
3,Installation de chauffage simple,Un seul système d'ECS sans solaire,installation individuelle,individuel,ecs,Électricité,Ballon électrique à accumulation vertical Caté...,False,moyenne,insuffisante,...,C,F,Radiateur électrique à accumulation,très bonne,Électricité,<NA>,<NA>,<NA>,housenumber,6
4,Installation de chauffage simple,Un seul système d'ECS sans solaire,installation individuelle,individuel,ecs,Électricité,Ballon électrique à accumulation vertical Caté...,True,bonne,insuffisante,...,C,F,Radiateur électrique à accumulation,très bonne,Électricité,<NA>,<NA>,<NA>,housenumber,6


In [7]:
categ_cols_init = set(df_clean_v1.select_dtypes(include='string').columns)
categ_cols = categ_cols_init.copy()

# decomposer les colonnes par groupes pour appliquer encodage
# on traite au cas par cas pour identifier les cols à ordinal_encoder et on labelisera tout le reste
# observer la data

In [ ]:
# qualite cols ademe
get_qualite_col_ademe = lambda data: list(filter(lambda c: ('qualite' in c) and ('ademe' in c), data))

for _ in df_clean_v1[get_qualite_col_ademe(df_clean_v1)]:
    print(df_clean_v1[_].value_counts(dropna=False))

print(df_clean_v1['indicateur_confort_ete_ademe'].value_counts())

# mapping ordinal encoder for quality cols
mapping_oe_qualite_col_ademe = {'insuffisante': 1, 'moyenne': 2, 'bonne': 3, 'très bonne': 4, '<NA>': -1}
to_oe_qualite_col_ademe = [
        'qualite_isolation_menuiseries_ademe',
        'qualite_isolation_murs_ademe',
        'qualite_isolation_plancher_haut_comble_perdu_ademe',
        'qualite_isolation_enveloppe_ademe',
        'qualite_isolation_plancher_bas_ademe',
        'qualite_isolation_plancher_haut_toit_terrase_ademe',
        'indicateur_confort_ete_ademe'
    ]

In [9]:
categ_cols -= set(to_oe_qualite_col_ademe)

In [10]:
df_clean_v1.select_dtypes(include='string')[list(categ_cols)].head() 

,type_energie_principale_chauffage_ademe,etiquette_dpe_ademe,type_energie_ndeg2_ademe,classe_inertie_batiment_ademe,etiquette_ges_ademe,usage_generateur_ecs_ndeg1_ademe,categorie_enr_ademe,logement_traversant_0_1_ademe,inertie_lourde_0_1_ademe,type_energie_principale_ecs_ademe,...,configuration_installation_chauffage_ndeg1_ademe,protection_solaire_exterieure_0_1_ademe,usage_generateur_ndeg1_installation_ndeg1_ademe,isolation_toiture_0_1_ademe,configuration_installation_ecs_ademe,methode_application_dpe_ademe,type_installation_ecs_general_ademe,type_energie_generateur_ndeg1_installation_ndeg1_ademe,type_generateur_ecs_ndeg1_ademe,type_generateur_ndeg1_installation_ndeg1_ademe
0,Gaz naturel,C,Électricité,Moyenne,C,chauffage + ecs,<NA>,True,False,Gaz naturel,...,Installation de chauffage simple,True,chauffage + ecs,False,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Chaudière gaz à condensation après 2015,Chaudière gaz à condensation après 2015
1,Électricité,D,<NA>,Moyenne,B,ecs,<NA>,False,False,Électricité,...,Installation de chauffage simple,False,chauffage,True,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
2,Gaz naturel,E,Électricité,Moyenne,D,ecs,<NA>,True,False,Électricité,...,Installation de chauffage simple,False,chauffage,False,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Ballon électrique à accumulation horizontal,Chaudière gaz basse température 2001-2015
3,Électricité,F,<NA>,Moyenne,C,ecs,<NA>,False,False,Électricité,...,Installation de chauffage simple,False,chauffage,True,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
4,Électricité,F,<NA>,Moyenne,C,ecs,<NA>,True,False,Électricité,...,Installation de chauffage simple,False,chauffage,True,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation


In [ ]:
# traiter les colonnes True/False

get_binary_cols_ademe = lambda data: list(filter(lambda c: ('0_1' in c) and ('ademe' in c), data))

for _ in df_clean_v1[get_binary_cols_ademe(df_clean_v1)]:
    print(df_clean_v1[_].value_counts(dropna=False))

# mapping binary cols ademe
mapping_binary_col_ademe = {'True': 1, 'False': 0, '<NA>': -1} # pertinent car on n'a pas le type bool donc si on labelisait, l'encoder pourrait inverser 0 et 1 selon la colonne 
                                                               # on peut forcer le mapping surout à cause des NA
to_binary_col_ademe = [
        'logement_traversant_0_1_ademe',
        'presence_brasseur_air_0_1_ademe',
        'protection_solaire_exterieure_0_1_ademe',
        'inertie_lourde_0_1_ademe',
        'isolation_toiture_0_1_ademe',
        'appartement_non_visite_0_1_ademe'
    ]

In [12]:
categ_cols -= set(to_binary_col_ademe)

In [14]:
df_clean_v1.select_dtypes(include='string')[list(categ_cols)].head() 

,type_energie_principale_chauffage_ademe,etiquette_dpe_ademe,type_energie_ndeg2_ademe,classe_inertie_batiment_ademe,etiquette_ges_ademe,usage_generateur_ecs_ndeg1_ademe,categorie_enr_ademe,type_energie_principale_ecs_ademe,type_energie_ndeg1_ademe,type_energie_generateur_ecs_ndeg1_ademe,...,type_batiment_ademe,type_installation_chauffage_ademe,configuration_installation_chauffage_ndeg1_ademe,usage_generateur_ndeg1_installation_ndeg1_ademe,configuration_installation_ecs_ademe,methode_application_dpe_ademe,type_installation_ecs_general_ademe,type_energie_generateur_ndeg1_installation_ndeg1_ademe,type_generateur_ecs_ndeg1_ademe,type_generateur_ndeg1_installation_ndeg1_ademe
0,Gaz naturel,C,Électricité,Moyenne,C,chauffage + ecs,<NA>,Gaz naturel,Gaz naturel,Gaz naturel,...,appartement,individuel,Installation de chauffage simple,chauffage + ecs,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Chaudière gaz à condensation après 2015,Chaudière gaz à condensation après 2015
1,Électricité,D,<NA>,Moyenne,B,ecs,<NA>,Électricité,Électricité,Électricité,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
2,Gaz naturel,E,Électricité,Moyenne,D,ecs,<NA>,Électricité,Gaz naturel,Électricité,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Ballon électrique à accumulation horizontal,Chaudière gaz basse température 2001-2015
3,Électricité,F,<NA>,Moyenne,C,ecs,<NA>,Électricité,Électricité,Électricité,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
4,Électricité,F,<NA>,Moyenne,C,ecs,<NA>,Électricité,Électricité,Électricité,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation


In [ ]:
# etiquettes ademe # dont DPE  # CLASSES A - G
get_etiquette_ademe_cols_ordinal = lambda data: list(filter(lambda c: ('etiquette' in c) and ('ademe' in c), data))

to_oe_etiquettes_ademe = ['etiquette_ges_ademe', 'etiquette_dpe_ademe']

for _ in to_oe_etiquettes_ademe:
    print(df_clean_v1[_].value_counts(dropna=False))

mapping_to_oe_etiquette_ademe = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, '<NA>': -1}

In [16]:
categ_cols -= set(to_oe_etiquettes_ademe)

In [17]:
df_clean_v1.select_dtypes(include='string')[list(categ_cols)].head() 

,type_energie_principale_chauffage_ademe,type_energie_ndeg2_ademe,classe_inertie_batiment_ademe,usage_generateur_ecs_ndeg1_ademe,categorie_enr_ademe,type_energie_principale_ecs_ademe,type_energie_ndeg1_ademe,type_energie_generateur_ecs_ndeg1_ademe,type_enedis_with_ban,type_emetteur_installation_chauffage_ndeg1_ademe,...,type_batiment_ademe,type_installation_chauffage_ademe,configuration_installation_chauffage_ndeg1_ademe,usage_generateur_ndeg1_installation_ndeg1_ademe,configuration_installation_ecs_ademe,methode_application_dpe_ademe,type_installation_ecs_general_ademe,type_energie_generateur_ndeg1_installation_ndeg1_ademe,type_generateur_ecs_ndeg1_ademe,type_generateur_ndeg1_installation_ndeg1_ademe
0,Gaz naturel,Électricité,Moyenne,chauffage + ecs,<NA>,Gaz naturel,Gaz naturel,Gaz naturel,housenumber,Radiateur bitube sans robinet thermostatique s...,...,appartement,individuel,Installation de chauffage simple,chauffage + ecs,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Chaudière gaz à condensation après 2015,Chaudière gaz à condensation après 2015
1,Électricité,<NA>,Moyenne,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
2,Gaz naturel,Électricité,Moyenne,ecs,<NA>,Électricité,Gaz naturel,Électricité,housenumber,Radiateur monotube sans robinet thermostatique...,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Ballon électrique à accumulation horizontal,Chaudière gaz basse température 2001-2015
3,Électricité,<NA>,Moyenne,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
4,Électricité,<NA>,Moyenne,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation


In [18]:
# 
to_oe_classe_inertie_bat_ademe = ['classe_inertie_batiment_ademe']
mapping_oe_classe_inertie_bat_ademe = {'Légère': 1, 'Moyenne': 2, 'Lourde': 3, 'Très lourde': 4, '<NA>': -1}

df_clean_v1['classe_inertie_batiment_ademe'].value_counts(dropna=False)

classe_inertie_batiment_ademe
Légère         127686
Moyenne        116712
Lourde          85261
Très lourde     36407
<NA>             3015
Name: count, dtype: Int64

In [19]:
categ_cols -= set(to_oe_classe_inertie_bat_ademe)

In [20]:
df_clean_v1.select_dtypes(include='string')[list(categ_cols)].head() 

,type_energie_principale_chauffage_ademe,type_energie_ndeg2_ademe,usage_generateur_ecs_ndeg1_ademe,categorie_enr_ademe,type_energie_principale_ecs_ademe,type_energie_ndeg1_ademe,type_energie_generateur_ecs_ndeg1_ademe,type_enedis_with_ban,type_emetteur_installation_chauffage_ndeg1_ademe,type_installation_chauffage_ndeg1_ademe,...,type_batiment_ademe,type_installation_chauffage_ademe,configuration_installation_chauffage_ndeg1_ademe,usage_generateur_ndeg1_installation_ndeg1_ademe,configuration_installation_ecs_ademe,methode_application_dpe_ademe,type_installation_ecs_general_ademe,type_energie_generateur_ndeg1_installation_ndeg1_ademe,type_generateur_ecs_ndeg1_ademe,type_generateur_ndeg1_installation_ndeg1_ademe
0,Gaz naturel,Électricité,chauffage + ecs,<NA>,Gaz naturel,Gaz naturel,Gaz naturel,housenumber,Radiateur bitube sans robinet thermostatique s...,installation individuelle,...,appartement,individuel,Installation de chauffage simple,chauffage + ecs,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Chaudière gaz à condensation après 2015,Chaudière gaz à condensation après 2015
1,Électricité,<NA>,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,installation individuelle,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
2,Gaz naturel,Électricité,ecs,<NA>,Électricité,Gaz naturel,Électricité,housenumber,Radiateur monotube sans robinet thermostatique...,installation individuelle,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Gaz naturel,Ballon électrique à accumulation horizontal,Chaudière gaz basse température 2001-2015
3,Électricité,<NA>,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,installation individuelle,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation
4,Électricité,<NA>,ecs,<NA>,Électricité,Électricité,Électricité,housenumber,Radiateur électrique à accumulation,installation individuelle,...,appartement,individuel,Installation de chauffage simple,chauffage,Un seul système d'ECS sans solaire,dpe appartement individuel,individuel,Électricité,Ballon électrique à accumulation vertical Caté...,Radiateur électrique à accumulation


In [21]:
to_oe_periode_construction = ['periode_construction_ademe']
mapping_periode_construction = {
    "avant 1948": 1,
    "1948-1974": 2,
    "1975-1977": 3,
    "1978-1982": 4,
    "1983-1988": 5,
    "1989-2000": 6,
    "2001-2005": 7,
    "2006-2012": 8,
    "2013-2021": 9,
    "après 2021": 10
    }

df_clean_v1['periode_construction_ademe'].value_counts(dropna=False).sort_index()
# il y a un ordre qui n'est pas respecté quand on sort l'index

periode_construction_ademe
1948-1974      67497
1975-1977       9457
1978-1982       8131
1983-1988       9412
1989-2000      20057
2001-2005       3479
2006-2012       2538
2013-2021       4452
après 2021       588
avant 1948    243470
Name: count, dtype: Int64

In [22]:
categ_cols -= set(to_oe_periode_construction)

In [24]:
ov = {}
for _ in df_clean_v1.select_dtypes(include='string')[list(categ_cols)]:
    ov.update({_ : df_clean_v1[_].value_counts(dropna=False).to_dict()})

ov

{'type_energie_principale_chauffage_ademe': {'Électricité': 165167,
  'Gaz naturel': 137375,
  'Réseau de Chauffage urbain': 50706,
  <NA>: 10189,
  'Fioul domestique': 5058,
  'GPL': 455,
  'Bois – Bûches': 83,
  'Bois – Granulés (pellets) ou briquettes': 26,
  'Propane': 8,
  'Charbon': 5,
  'Bois – Plaquettes forestières': 4,
  'Butane': 4,
  'Réseau de Froid Urbain': 1},
 'type_energie_ndeg2_ademe': {'Électricité': 194409,
  <NA>: 164769,
  'Gaz naturel': 7874,
  'Réseau de Chauffage urbain': 1449,
  'Fioul domestique': 299,
  'Bois – Bûches': 213,
  'GPL': 32,
  'Bois – Granulés (pellets) ou briquettes': 21,
  "Électricité d'origine renouvelable utilisée dans le bâtiment": 7,
  'Bois – Plaquettes forestières': 4,
  'Bois – Plaquettes d’industrie': 3,
  'Réseau de Froid Urbain': 1},
 'usage_generateur_ecs_ndeg1_ademe': {'ecs': 198685,
  'chauffage + ecs': 151860,
  <NA>: 12337,
  'chauffage': 6199},
 'categorie_enr_ademe': {<NA>: 323553,
  'réseau de chaleur ou de froid vertueux': 

### Application encodage dataframe v1 (recap)

In [7]:
cols_qualite_ademe_to_oe = [
    'qualite_isolation_menuiseries_ademe',
    'qualite_isolation_murs_ademe',
    'qualite_isolation_plancher_haut_comble_perdu_ademe',
    'qualite_isolation_enveloppe_ademe',
    'qualite_isolation_plancher_bas_ademe',
    'qualite_isolation_plancher_haut_toit_terrase_ademe',
    'indicateur_confort_ete_ademe'
    ]

cols_0_1_ademe_to_oe = [
    'logement_traversant_0_1_ademe',
    'presence_brasseur_air_0_1_ademe',
    'protection_solaire_exterieure_0_1_ademe',
    'inertie_lourde_0_1_ademe',
    'isolation_toiture_0_1_ademe',
    'appartement_non_visite_0_1_ademe'
    ]

cols_etiquettes_ademe_to_oe = ['etiquette_ges_ademe', 'etiquette_dpe_ademe']
col_classe_inertie_bat_ademe_to_oe = ['classe_inertie_batiment_ademe']
col_periode_construction_to_oe = ['periode_construction_ademe']

# ---------
get_col_mapping = lambda colname, mapping: {'col': colname, 'mapping': mapping}

mapping_qualite_ademe_oe = {'insuffisante': 1, 'moyenne': 2, 'bonne': 3, 'très bonne': 4, '<NA>': -1}
mapping_0_1_ademe_oe = {'True': 1, 'False': 0, '<NA>': -1} 
mapping_etiquettes_ademe_oe = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, '<NA>': -1}
mapping_classe_inertie_bat_ademe_oe = {'Légère': 1, 'Moyenne': 2, 'Lourde': 3, 'Très lourde': 4, '<NA>': -1}
mapping_periode_construction_oe = {
    "avant 1948": 1,
    "1948-1974": 2,
    "1975-1977": 3,
    "1978-1982": 4,
    "1983-1988": 5,
    "1989-2000": 6,
    "2001-2005": 7,
    "2006-2012": 8,
    "2013-2021": 9,
    "après 2021": 10,
    '<NA>': -1
    }

# ---- ordinal encoder all in one

cols_to_ordinal_encode = cols_qualite_ademe_to_oe + cols_0_1_ademe_to_oe + cols_etiquettes_ademe_to_oe + col_classe_inertie_bat_ademe_to_oe + col_periode_construction_to_oe

full_oe_mapping = [get_col_mapping(c, mapping_qualite_ademe_oe) for c in cols_qualite_ademe_to_oe]
full_oe_mapping += [get_col_mapping(c, mapping_0_1_ademe_oe) for c in cols_0_1_ademe_to_oe]
full_oe_mapping += [get_col_mapping(c, mapping_etiquettes_ademe_oe) for c in cols_etiquettes_ademe_to_oe]
full_oe_mapping += [get_col_mapping(c, mapping_classe_inertie_bat_ademe_oe) for c in col_classe_inertie_bat_ademe_to_oe]
full_oe_mapping += [get_col_mapping(c, mapping_periode_construction_oe) for c in col_periode_construction_to_oe]

other_cols_to_label_encode = list(set(df_clean_v1.select_dtypes(include='string').columns) - set(cols_to_ordinal_encode))


In [7]:
get_n_classes = lambda data, colname: len(data[colname].unique())

In [12]:
numerisation_pipeline = ColumnTransformer(
        transformers=[
            # ('log_transformer', LogTransformer(columns=quant_features_names), quant_features_names),
            ('ordinal_encoding', ce.OrdinalEncoder(mapping=full_oe_mapping), cols_to_ordinal_encode),
            ('label_encoding', ce.BinaryEncoder(base=112), other_cols_to_label_encode)
        ], 
        remainder='passthrough',
        verbose_feature_names_out=False
    )

df_clean_v1_copy = df_clean_v1.copy()

df_clean_v1_copy = numerize(df_clean_v1_copy, numerisation_pipeline)
df_clean_v1_copy.head()

,qualite_isolation_menuiseries_ademe,qualite_isolation_murs_ademe,qualite_isolation_plancher_haut_comble_perdu_ademe,qualite_isolation_enveloppe_ademe,qualite_isolation_plancher_bas_ademe,qualite_isolation_plancher_haut_toit_terrase_ademe,indicateur_confort_ete_ademe,logement_traversant_0_1_ademe,presence_brasseur_air_0_1_ademe,protection_solaire_exterieure_0_1_ademe,...,conso_chauffage_depensier_e_primaire_ademe,score_ban_ademe,surface_habitable_immeuble_ademe,annee_construction_ademe,nombre_niveau_immeuble_ademe,consommation_annuelle_totale_de_l_adresse_mwh_enedis_with_ban,consommation_annuelle_moyenne_de_la_commune_mwh_enedis_with_ban,lon_enedis_with_ban,lat_enedis_with_ban,consommation_annuelle_moyenne_par_logement_de_l_adresse_kwh_enedis_with_ban
0,4,1.0,4.0,1,4.0,-1.0,-1.0,1.0,0.0,1.0,...,20573.599609,0.72,2587.0,1947.0,7.0,78.292,2.686,2.328325,48.84874,4350.0
1,1,1.0,1.0,4,4.0,-1.0,-1.0,0.0,0.0,0.0,...,2312.300049,0.72,2587.0,1947.0,7.0,78.292,2.686,2.328325,48.84874,4350.0
2,1,1.0,4.0,1,4.0,-1.0,-1.0,1.0,0.0,0.0,...,41476.398438,0.72,2587.0,1947.0,7.0,78.292,2.686,2.328325,48.84874,4350.0
3,2,1.0,1.0,3,4.0,-1.0,-1.0,0.0,0.0,0.0,...,4481.700195,0.72,2587.0,1947.0,7.0,78.292,2.686,2.328325,48.84874,4350.0
4,3,1.0,3.0,4,4.0,-1.0,-1.0,1.0,0.0,0.0,...,4594.100098,0.72,2587.0,1947.0,7.0,78.292,2.686,2.328325,48.84874,4350.0


In [14]:
df_clean_v1_copy.dtypes.value_counts()

float32    62
int32      27
Name: count, dtype: int64

### Encodage dataframe v2 (use correlation with target selection)

In [26]:
df = load_parquet_data('../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet')

pipeline = Nettoyage(df)

pipeline.run(use_target_correlation_selection=True)
df_clean_v2 = pipeline.df

del df

Loading parquet data from : ../ressources/data/2_intermediary/enedis_ban_ademe_extract_PARIS_2022.parquet..
-> Delete cols mano..
-> Delete NaN cols..
Le dataframe contient initialement 250 colonnes et 464576 lignes.
Start processing : columns identification..
Il y a 76 colonnes vides à au moins, 90.0%, 55 colonnes avec une entropie de 0 ou 1 et 13 colonnes inutiles
Start processing : columns deleting ...
Il reste 146 colonnes et 375231 lignes.
-> Start object columns auto casting..
End casting
-> Fill pd.NA for float dtypes..
Done fillnan !
Les colonnes suivantes ont une corrélation supérieure à 0.9 : ['emission_ges_chauffage_energie_ndeg1_ademe', 'conso_chauffage_depensier_installation_chauffage_ndeg1_ademe', 'conso_chauffage_installation_chauffage_ndeg1_ademe', 'emission_ges_5_usages_energie_ndeg1_ademe', 'cout_refroidissement_depensier_ademe', 'emission_ges_eclairage_ademe', 'conso_e_finale_depensier_generateur_ecs_ndeg1_ademe', 'cout_chauffage_energie_ndeg1_ademe', 'conso_chauffag

In [27]:
df_clean_v2.dtypes.value_counts()

float32           50
string[python]    39
Name: count, dtype: int64

In [30]:
assert all(_ in df_clean_v2 for _ in  cols_to_ordinal_encode + other_cols_to_label_encode )

In [21]:
numerisation_pipeline_v2 = ColumnTransformer(
        transformers=[
            # ('log_transformer', LogTransformer(columns=quant_features_names), quant_features_names),
            ('ordinal_encoding', ce.OrdinalEncoder(mapping=full_oe_mapping), cols_to_ordinal_encode),
            ('label_encoding', ce.BinaryEncoder(base=112), other_cols_to_label_encode)
        ], 
        remainder='passthrough',
        verbose_feature_names_out=False
    )

In [22]:
df_clean_v2_copy = df_clean_v2.copy()

df_clean_v2_copy = numerize(df_clean_v2_copy, numerisation_pipeline_v2)

In [25]:
df_clean_v2_copy.dtypes.value_counts()

float32    62
int32      27
Name: count, dtype: int64